In [ ]:
# Copyright 2023 Google LLC
# SPDX-License-Identifier: Apache-2.0

# Controlled state preparation using rotations

Adapted from Qualtran v0.7.0, Copyright 2023 Google LLC, Apache-2.0.

In [ ]:
from IPython.display import SVG, display
from examples.algorithms.field_addition_catalog import field_signature_svg
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg
from examples.algorithms.rotation_preparation_catalog import RotationPreparationExample, rotation_signature
from dataclasses import replace
from random import Random
import torch
from flagquantum.compiler.registers.phase.phase_gradient import phase_gradient_state_ir
from flagquantum.core.ir import CircuitIR
from flagquantum.dynamic import DynamicCircuit
from flagquantum.runtime.dynamic.execution import run_dynamic
torch.set_num_threads(1)

The operation uses phase kickback from a reusable negative-exponent gradient. Supply the gradient alongside the target. The tutorial initializes the gradient with the native phase-gradient state circuit.

- [State preparation using rotations](https://arxiv.org/abs/1812.00954)

## Example of use

Generate the original seeded three-qubit complex state and use four bits for angle precision.

In [ ]:
rng = Random(137)
phase_width, target_width = 4, 3
raw = torch.tensor(
    [complex(rng.uniform(-1, 1), rng.uniform(-1, 1)) for _ in range(1 << target_width)],
    dtype=torch.complex128,
)
state = raw / raw.norm()

## Build the operation

The coefficients must have unit norm and power-of-two length. The phase width sets the quantization resolution. The graph shows the three amplitude lookups and the final phase lookup.

In [ ]:
op = RotationPreparationExample(tuple(state.tolist()), phase_width)
display(SVG(field_signature_svg([("Rotation preparation", op.signature, {})])))
graph = op.call_graph()
display(SVG(render_call_graph_svg(export_call_graph_dot(graph.root, graph.direct_calls))))

## Append and execute

Prepare the gradient, append the operation, then erase the restored gradient. Each trajectory checks that the gradient and clean workspace return to zero.

In [ ]:
def run_example(*operations):
    op = operations[0]
    circuit = DynamicCircuit(max(example.qubit_count for example in operations), dtype=torch.complex128)
    if op.controls:
        circuit.h(0)
    circuit.compose(CircuitIR(circuit.n_qubits, phase_gradient_state_ir(op.gradient, exponent=-1)))
    for example in operations:
        circuit.append(example)
    circuit.compose(CircuitIR(circuit.n_qubits, phase_gradient_state_ir(op.gradient, exponent=-1, inverse=True)))
    result = run_dynamic(circuit, shots=3, seed=27, strategy="trajectory")
    amplitudes = result.final_states.reshape(3, 1 << (op.controls + op.target_width), -1)
    torch.testing.assert_close(amplitudes[:, :, 1:], torch.zeros_like(amplitudes[:, :, 1:]), atol=2e-12, rtol=0)
    output = amplitudes[0, :, 0]
    torch.testing.assert_close(amplitudes[:, :, 0], output.expand(3, -1), atol=2e-12, rtol=0)
    return output

prepared = run_example(op)

Compare the prepared and requested complex coefficients. Quantizing the angles can change both magnitudes and phases.

In [ ]:
overlap = abs(torch.vdot(state, prepared))
assert overlap > 0.95
display(float(overlap))
display(list(zip(state.tolist(), prepared.tolist())))

## Controlled state preparation

Put one control into $|+\rangle$. The $|0\rangle$ branch leaves the target in $|0\rangle$. The $|1\rangle$ branch contains the prepared state, including its relative phase.

In [ ]:
controlled = replace(op, controls=1)
controlled_state = run_example(controlled)
zero = torch.zeros_like(state)
zero[0] = 1
ideal = torch.cat((zero, state)) / 2**0.5
expected = torch.cat((zero, prepared)) / 2**0.5
torch.testing.assert_close(controlled_state, expected, atol=2e-12, rtol=0)

Compare both control branches together with the requested coherent output.

In [ ]:
display(float(abs(torch.vdot(ideal, controlled_state))))
display(list(zip(ideal.tolist(), controlled_state.tolist())))

## Using the adjoint

Applying the matched uncompute after preparation should restore $|0\rangle$ on the target. The gradient is then erased.

In [ ]:
restored = run_example(op, replace(op, uncompute=True))
torch.testing.assert_close(restored, zero, atol=2e-12, rtol=0)
display(restored.tolist())